# Module 07 · Cross-cohort meta-analysis

Does the burden trajectory replicate?

Module 06 fits one burden GLMM per cell type per cohort and writes
`burden_glmm_<condition>.csv`. This module reads those files — nothing else —
and pools them under a DerSimonian-Laird random-effects model.

Reading per-cohort result tables rather than pooling raw cells is deliberate.
The cohorts differ in tissue handling, sequencing depth and donor composition,
so a single model fit across them would confound cohort with effect. Pooling
the *estimates* keeps each cohort's model inside its own data and lets the
between-cohort variance be estimated rather than assumed away.

**What comes out:** a pooled OR per cell type, a 95% CI, a BH-adjusted p across
cell types, and the two heterogeneity statistics — Q with its p, and I².

| Section | |
|---|---|
| 01 | config — cohort registry, effect selection, path reconstruction |
| 02 | load and inspect per-cohort results, coverage table, SE integrity |
| 03 | DerSimonian-Laird pooling |
| 04 | forest plot — per-cohort markers plus pooled diamond |

**Prerequisite.** Module 06 must have been run once per cohort, with `DATASET`
set to that cohort's key each time. This module reconstructs the module 06
output paths from `COHORTS` and will report `MISSING` for any it cannot find.

> **Note.** The paths this module reconstructs still contain
> `module_03_burden_modeling/` — that is the directory module 06 writes to,
> inherited from the source notebook's older numbering. The two agree; only the
> notebook filenames were renumbered.


---
## 01 · Config

**Why.** The cohort registry is the whole configuration. Each entry names a
cohort and the `DATASET` key that module 06 was run under, and the path is
reconstructed from module 06's own directory scheme — so the two modules cannot
drift apart as long as the keys match.

`_env()` is the only addition to the source; it replaces the hardcoded root.

**Set before running:** `STUDY_TYPE`, `COHORTS`, `MIN_COHORTS`.

**Two things to know about this cell as it stands:**

1. `SLUG` is `f"burden_glmm_{CONDITION_TAG}"` — the string `burden_glmm` is
   written in literally, not taken from `EFFECT`. Changing `EFFECT` to
   `"susceptibility_glmm"` changes the output filenames and the axis label but
   **not** the file that gets read, so you would pool burden estimates and save
   them under a susceptibility name. To pool susceptibility, change `SLUG` too.
2. `MIN_COHORTS = 1` admits a cell type present in only one cohort. With k = 1
   there is no pooling to do — τ² is 0 by construction, I² is undefined, and the
   "pooled" row is just that cohort's estimate. It still receives a p-value and
   enters the FDR correction alongside genuinely pooled rows.

In [ ]:
# -----------------------------------------------------------------------------
# Path resolution - the only addition to the source config
# -----------------------------------------------------------------------------
#   SENESCENCE_DATA : analysis root, the same one module 07 wrote under
# See .env.example.
import os


def _env(name):
    v = os.environ.get(name)
    if not v:
        raise RuntimeError(
            f"{name} is not set. Copy .env.example to .env, edit the paths, "
            f"and source it before starting the kernel.")
    return v.rstrip("/")


print(f"  SENESCENCE_DATA -> {_env('SENESCENCE_DATA')}")

In [ ]:
# =============================================================================
# CELL 1 — CONFIG + SETUP  (paths mirror Module 3's own scheme)
# =============================================================================
from pathlib import Path
import numpy as np, pandas as pd
pd.set_option("display.max_columns", None); pd.set_option("display.width", 200)

STUDY_TYPE    = "aging"
EFFECT        = "burden_glmm"
EFFECT_LABEL  = "Burden OR / decade"
CONDITION_TAG = STUDY_TYPE
META_METHOD   = "DL"
FDR_THRESHOLD = 0.05
MIN_COHORTS   = 1

# ── reconstruct Module-3 output paths ─────────────────────────────────────────
#   {SCRATCH}/{TISSUE}/module_03_burden_modeling/{CONDITION_SUBPATH}/{DATASET}/results/{slug}.csv
SCRATCH = _env("SENESCENCE_DATA")
TISSUE  = "brain"
IS_AGING = (STUDY_TYPE == "aging")
DISEASE  = None                                   # set if STUDY_TYPE=="disease"
CONDITION_SUBPATH = STUDY_TYPE if IS_AGING else f"{STUDY_TYPE}/{DISEASE}"
SLUG = f"burden_glmm_{CONDITION_TAG}"             # matches save_table slug in §9.3c

def m3_results(dataset_key):
    return Path(f"{SCRATCH}/{TISSUE}/module_03_burden_modeling/"
                f"{CONDITION_SUBPATH}/{dataset_key}/results/{SLUG}.csv")

# cohorts: display name + the §9.3c DATASET key used when that run was saved
COHORTS = {
    "psychad":     {"name": "PsychAD",     "dataset_key": "psychad_aging"},
    "psychencode": {"name": "PsychEncode", "dataset_key": "psychencode"},
}
for cfg in COHORTS.values():
    cfg["path"] = m3_results(cfg["dataset_key"])

CELL_TYPE_COLORS = {
    "Excitatory":"#0072B2","Inhibitory":"#E69F00","Astrocyte":"#009E73",
    "Oligodendrocyte":"#56B4E9","Microglia":"#D55E00","OPC":"#CC79A7",
    "Endothelial":"#7F7F7F","Pericyte":"#999999",
}
OUTDIR = Path("results"); OUTDIR.mkdir(parents=True, exist_ok=True)
FIGDIR = Path("figures"); FIGDIR.mkdir(parents=True, exist_ok=True)

print("="*72); print(f"META · {STUDY_TYPE} · {EFFECT} · {META_METHOD}"); print("="*72)
for cid, cfg in COHORTS.items():
    print(f"  {cfg['name']:14s} ← {cfg['path']}  [{'exists' if cfg['path'].exists() else 'MISSING'}]")

---
## 02 · Load and inspect

**Why.** Nothing is pooled until the inputs have been looked at. Three checks
run here, in order.

**Required columns.** A cohort file must carry `Cell_Type`, `beta` and `se`. A
file missing any of them is dropped with a printed `✗` rather than being
silently partially used.

**SE integrity.** `se <= 0`, non-finite, or `se > 5` marks a row as suspect.
Those are the signature of a `glmer` fit with a degenerate Hessian — the point
estimate may look reasonable while the SE is meaningless. Because inverse-
variance weighting means a tiny SE dominates the pool, an undetected bad SE
does more damage than a missing row. Flagged rows are excluded from the pooling
in section 03.

**Coverage table.** Cell type × cohort, showing which combinations exist and
which have a usable SE. Read this before the forest — a cell type present in
one cohort is not replicated, whatever its pooled p-value says.

In [ ]:
# =============================================================================
# CELL 2 — LOAD + INSPECT per-cohort results (no pooling yet)
# =============================================================================
print("="*72); print("LOAD per-cohort §9.3c results"); print("="*72)

REQUIRED = {"Cell_Type", "beta", "se"}     # minimum needed to pool

def load_cohort(cid, cfg):
    p = Path(cfg["path"])
    if not p.exists():
        print(f"  ✗ {cfg['name']}: file not found"); return None
    df = pd.read_csv(p)
    print(f"\n  ── {cfg['name']} ──  ({p.name})")
    print(f"     columns: {list(df.columns)}")
    print(f"     rows: {len(df)}")
    missing = REQUIRED - set(df.columns)
    if missing:
        print(f"     ✗ MISSING required columns: {missing}"); return None
    df = df.copy(); df["cohort"] = cid; df["cohort_name"] = cfg["name"]
    # integrity: flag non-finite / degenerate SEs (degenerate-Hessian fits)
    bad = ~np.isfinite(df["se"]) | (df["se"] <= 0) | (df["se"] > 5)
    if bad.any():
        print(f"     ⚠ {int(bad.sum())} row(s) with suspect SE (likely non-converged glmer): "
              f"{df.loc[bad,'Cell_Type'].tolist()}")
    df["se_ok"] = ~bad
    print(df[["Cell_Type","beta","se","se_ok"] + (["p"] if "p" in df.columns else [])]
          .assign(OR=lambda d: np.exp(d.beta).round(3)).to_string(index=False))
    return df

frames = [d for d in (load_cohort(k, v) for k, v in COHORTS.items()) if d is not None]
assert frames, "no cohort results loaded — fix COHORTS paths"
long = pd.concat(frames, ignore_index=True)

# coverage: which cell types appear in which cohorts, and how many have usable SE
print("\n" + "="*72); print("COVERAGE (cell type × cohort, ✓=usable SE)"); print("="*72)
cov = (long.assign(mark=lambda d: np.where(d.se_ok, "✓", "⚠"))
       .pivot_table(index="Cell_Type", columns="cohort_name", values="mark", aggfunc="first")
       .fillna("—"))
print(cov.to_string())
n_bad = int((~long["se_ok"]).sum())
print(f"\n  total cohort×CT rows: {len(long)} | usable: {int(long['se_ok'].sum())} | suspect SE: {n_bad}")
if n_bad: print("  ⚠ suspect-SE rows will mis-weight the meta — consider re-running those cohorts as quasi-binomial GLM")

---
## 03 · DerSimonian-Laird pooling

**Why.** Random effects rather than fixed. A fixed-effect pool assumes every
cohort estimates the *same* underlying value and that all disagreement is
sampling noise. Across cohorts that differ in dissection, depth and donor
composition, that assumption is not credible; DL estimates the between-cohort
variance and widens the interval by it.

**Formula.** Per cell type, with cohort estimates `b_i`, standard errors
`s_i`, and `v_i = s_i^2`:

```
fixed-effect weights     w_i    = 1 / v_i
fixed-effect pool        b_FE   = sum(w_i * b_i) / sum(w_i)
Cochran's Q              Q      = sum(w_i * (b_i - b_FE)^2)
                         df     = k - 1

scaling constant         C      = sum(w_i) - sum(w_i^2) / sum(w_i)
between-cohort variance  tau2   = max(0, (Q - df) / C)

random-effect weights    W_i    = 1 / (v_i + tau2)
pooled estimate          b_RE   = sum(W_i * b_i) / sum(W_i)
pooled standard error    se_RE  = sqrt(1 / sum(W_i))

inconsistency            I2     = max(0, (Q - df) / Q) * 100
```

Inference is a Wald z on `b_RE / se_RE`; Q is tested against chi-square on
`k - 1` df. BH across cell types on the pooled p. Estimates are on the log-odds
scale throughout and exponentiated only for display.

**Reading the heterogeneity statistics at k = 2.** With two cohorts, Q has one
degree of freedom. I² near 0 means the two estimates agree; it does not mean
heterogeneity has been ruled out, because there is almost no power to detect it.
τ² is estimated from a single deviation and is correspondingly unstable. Treat
low I² at k = 2 as "the two cohorts did not visibly disagree", not as evidence
of homogeneity.

**Display.** Pooled table sorted by OR, saved to
`meta_{EFFECT}_{CONDITION_TAG}.csv`.

In [ ]:
# =============================================================================
# CELL 3 — DerSimonian-Laird random-effects pooling, per cell type
# =============================================================================
print("="*72); print("DL RANDOM-EFFECTS POOLING (per cell type)"); print("="*72)

from scipy import stats

def dl_pool(betas, ses):
    """DerSimonian-Laird random-effects pooling. Returns dict of pooled stats."""
    betas = np.asarray(betas, float); ses = np.asarray(ses, float)
    k = len(betas); v = ses**2
    w_fe = 1.0 / v
    beta_fe = np.sum(w_fe*betas) / np.sum(w_fe)
    Q = float(np.sum(w_fe * (betas - beta_fe)**2))
    df = k - 1
    if k > 1:
        C = np.sum(w_fe) - np.sum(w_fe**2)/np.sum(w_fe)
        tau2 = max(0.0, (Q - df) / C) if C > 0 else 0.0
        I2 = max(0.0, (Q - df) / Q) * 100 if Q > 0 else 0.0
    else:
        tau2, I2 = 0.0, np.nan
    w_re = 1.0 / (v + tau2)
    beta_re = np.sum(w_re*betas) / np.sum(w_re)
    se_re   = np.sqrt(1.0 / np.sum(w_re))
    z = beta_re / se_re
    p = 2 * stats.norm.sf(abs(z))
    Q_p = float(stats.chi2.sf(Q, df)) if df > 0 else np.nan
    return dict(k=k, pooled_beta=beta_re, pooled_se=se_re,
                ci_lo=beta_re-1.96*se_re, ci_hi=beta_re+1.96*se_re,
                z=z, p=p, Q=Q, Q_p=Q_p, tau2=tau2, I2=I2)

use = long[long["se_ok"]].copy()
rows = []
for ct, g in use.groupby("Cell_Type"):
    if g["cohort"].nunique() < MIN_COHORTS:
        continue
    pooled = dl_pool(g["beta"].values, g["se"].values)
    rows.append(dict(
        Cell_Type=ct, n_cohorts=pooled["k"],
        cohort_names=list(g["cohort_name"]),
        cohort_betas=list(g["beta"]), cohort_ses=list(g["se"]),
        **{k: pooled[k] for k in ["pooled_beta","pooled_se","ci_lo","ci_hi","p","Q","Q_p","tau2","I2"]},
    ))
meta = pd.DataFrame(rows)
# BH across cell types on the pooled p
from statsmodels.stats.multitest import multipletests
meta["p_adj"] = multipletests(meta["p"], method="fdr_bh")[1]
meta["Significant"] = meta["p_adj"] < FDR_THRESHOLD
# OR-scale display
for c0, c1 in [("pooled_beta","pooled_or"),("ci_lo","ci_lower_or"),("ci_hi","ci_upper_or")]:
    meta[c1] = np.exp(meta[c0])
meta = meta.sort_values("pooled_or", ascending=False).reset_index(drop=True)

print(meta.assign(
        pooled_OR=lambda d: d.pooled_or.round(3),
        CI=lambda d: "["+d.ci_lower_or.round(2).astype(str)+", "+d.ci_upper_or.round(2).astype(str)+"]",
        p=lambda d: d.p.map(lambda x: f"{x:.3f}"),
        FDR=lambda d: d.p_adj.map(lambda x: f"{x:.3f}"),
        I2=lambda d: d.I2.round(0).astype("Int64").astype(str)+"%",
        het_p=lambda d: d.Q_p.round(2))
      [["Cell_Type","n_cohorts","pooled_OR","CI","p","FDR","I2","het_p"]].to_string(index=False))
meta.to_csv(OUTDIR/f"meta_{EFFECT}_{CONDITION_TAG}.csv", index=False)
print(f"\n  ✓ saved → meta_{EFFECT}_{CONDITION_TAG}.csv")

---
## 04 · Forest plot

**Why.** The table alone hides whether a pooled estimate is a real convergence
or an average of two cohorts pointing opposite ways. Plotting the per-cohort
intervals alongside the pooled diamond makes that visible at a glance.

**Reads.** One marker per cohort per row, vertically offset, with its own
95% CI; the pooled random-effects estimate as a diamond whose width is the
pooled CI. Estimates outside the axis range are clipped and drawn with an
arrowhead rather than dropped.

**Columns.** Per-cohort OR (SE) · pooled OR [95% CI] · p · FDR · I².
I² is colour-coded — green below 25%, amber to 50%, orange to 75%, red above.

**Marks.** `★` FDR < 0.05, `*` p < 0.05.

**Writes.** `meta_{EFFECT}_{CONDITION_TAG}_forest.{pdf,svg,png}`.

Note that `OUTDIR` and `FIGDIR` are `results/` and `figures/` relative to the
working directory, not under the module 06 path scheme — this module writes
beside the notebook rather than beside its inputs.

In [ ]:
# =============================================================================
# CELL 4 — FOREST PLOT (table style) — meta burden trajectory
# =============================================================================
import matplotlib.pyplot as plt
import numpy as np, pandas as pd
from matplotlib.patches import Polygon
from matplotlib.lines import Line2D
print("="*72); print("FOREST (table style)"); print("="*72)

dfp = meta.sort_values("pooled_or", ascending=False).reset_index(drop=True)
n_rows = len(dfp)
cohort_list = list(COHORTS.keys())
NAME2CID = {v["name"]: k for k, v in COHORTS.items()}
COHORT_MARK = {COHORTS[c]["name"]: m for c, m in zip(cohort_list, ["s","o","^","D"])}
COHORT_COL  = {"PsychAD": "#08306B", "PsychEncode": "#67000D"}      # dark blue / dark red
for c in cohort_list:
    COHORT_COL.setdefault(COHORTS[c]["name"], "#444444")

fig = plt.figure(figsize=(9.4, max(3.2, n_rows*0.64+1.7))); ax = fig.add_subplot(111)
FH, FC, FD, FS = 9, 9, 8.5, 7.5

# ── column x-positions ────────────────────────────────────────────────────────
x = {"ct":0.005, "fL":0.15, "fR":0.42}
cohort_x = {}
_cols = [0.515, 0.625, 0.735, 0.845]
for i, c in enumerate(cohort_list):
    cohort_x[c] = _cols[i]
_next = _cols[len(cohort_list)-1] + 0.085
x["pooled"], x["p"], x["fdr"], x["i2"] = _next, _next+0.115, _next+0.175, _next+0.225

# ── log-OR scale with clipping ────────────────────────────────────────────────
allpe=[]
for _,r in dfp.iterrows(): allpe += list(r["cohort_betas"]) + [r["pooled_beta"]]
pmin,pmax=min(allpe),max(allpe); pad=max((pmax-pmin)*0.5,0.12); lo_b,hi_b=pmin-pad,pmax+pad
def to_x(v): v=np.clip(v,lo_b,hi_b); return x["fL"]+(v-lo_b)/(hi_b-lo_b)*(x["fR"]-x["fL"])
def clipL(v): return v<lo_b
def clipR(v): return v>hi_b
AR=0.005
OFF = [-0.15, 0.15] if len(cohort_list)==2 else list(np.linspace(-0.18,0.18,len(cohort_list)))

y_first=n_rows-1; y_hdr=y_first+0.50; y_clbl=y_first+0.80; y_hline=y_hdr-0.10

# ── headers ───────────────────────────────────────────────────────────────────
for c in cohort_list:
    nm=COHORTS[c]["name"]
    ax.text(cohort_x[c], y_clbl, nm, fontsize=FS, style="italic", color=COHORT_COL[nm], ha="center", va="bottom")
ax.text(x["ct"], y_hdr, "Cell Type", fontsize=FH, fontweight="bold", ha="left", va="bottom")
for c in cohort_list:
    ax.text(cohort_x[c], y_hdr, "OR (SE)", fontsize=FH, fontweight="bold", ha="center", va="bottom")
ax.text(x["pooled"], y_hdr, "Pooled OR", fontsize=FH, fontweight="bold", ha="center", va="bottom")
ax.text(x["p"],   y_hdr, "p",   fontsize=FH, fontweight="bold", ha="right", va="bottom")
ax.text(x["fdr"], y_hdr, "FDR", fontsize=FH, fontweight="bold", ha="right", va="bottom")
ax.text(x["i2"],  y_hdr, "I²",  fontsize=FH, fontweight="bold", ha="right", va="bottom")
ax.plot([0,0.99],[y_hline,y_hline],"k-",lw=1,clip_on=False)

# ── rows ──────────────────────────────────────────────────────────────────────
for idx,row in dfp.iterrows():
    y=n_rows-idx-1
    if idx%2==0: ax.axhspan(y-0.5,y+0.5,xmin=0.004,xmax=0.996,color="#f7f7f7",zorder=0)
    ct=row["Cell_Type"]; sigF=row["Significant"]; sigP=row["p"]<0.05
    ctc=CELL_TYPE_COLORS.get(ct,"#808080"); pre="★ " if sigF else ("* " if sigP else "")
    ax.text(x["ct"], y, pre+ct, fontsize=FC, fontweight="bold" if sigP else "normal",
            ha="left", va="center", color=ctc if sigF else "#333")

    # per-cohort markers (vertically offset) + OR / (SE) stacked text
    for j,(nm,b,se) in enumerate(zip(row["cohort_names"], row["cohort_betas"], row["cohort_ses"])):
        yo=y+OFF[j]; cl,cu=b-1.96*se,b+1.96*se; mk=COHORT_MARK.get(nm,"o"); cc=COHORT_COL.get(nm,"#888")
        ax.plot([to_x(cl),to_x(cu)],[yo,yo],color=cc,lw=1.4,alpha=0.8,zorder=2)
        if clipL(cl): ax.add_patch(Polygon([[to_x(cl),yo],[to_x(cl)+AR,yo+.06],[to_x(cl)+AR,yo-.06]],fc=cc,ec="none",alpha=.8))
        if clipR(cu): ax.add_patch(Polygon([[to_x(cu),yo],[to_x(cu)-AR,yo+.06],[to_x(cu)-AR,yo-.06]],fc=cc,ec="none",alpha=.8))
        ax.plot(to_x(b),yo,mk,ms=5.5,color=cc,mec="black",mew=0.4,zorder=3)
        cid=NAME2CID[nm]
        ax.text(cohort_x[cid], y+0.13, f"{np.exp(b):.3f}", fontsize=FD, ha="center", va="bottom")
        ax.text(cohort_x[cid], y-0.13, f"({se:.3f})", fontsize=FS, ha="center", va="top", color="#666")

    # pooled diamond + OR / [95% CI] stacked text
    rb,rs=row["pooled_beta"],row["pooled_se"]; rl,rh=rb-1.96*rs,rb+1.96*rs
    xr,xrl,xrh=to_x(rb),to_x(rl),to_x(rh)
    ax.plot([xrl,xrh],[y,y],color=ctc,lw=2.2,alpha=0.45,zorder=4)
    dwn,dhn=0.007,0.17
    ax.add_patch(Polygon([[xr-dwn,y],[xr,y-dhn],[xr+dwn,y],[xr,y+dhn]],fc=ctc,ec="black",lw=0.5,zorder=5,alpha=0.9))
    ax.text(x["pooled"], y+0.13, f"{row['pooled_or']:.3f}", fontsize=FD, ha="center", va="bottom",
            fontweight="bold" if sigP else "normal")
    ax.text(x["pooled"], y-0.13, f"[{row['ci_lower_or']:.2f}, {row['ci_upper_or']:.2f}]",
            fontsize=FS, ha="center", va="top", color="#666")

    # raw p
    pv=row["p"]; ptxt=f"{pv:.1e}" if pv<0.001 else (f"{pv:.3f}" if pv<0.01 else f"{pv:.2f}")
    ax.text(x["p"], y, ptxt, fontsize=FD, ha="right", va="center", fontweight="bold" if sigP else "normal")
    # FDR
    fv=row["p_adj"]; ftxt=f"{fv:.1e}" if fv<0.001 else (f"{fv:.3f}" if fv<0.01 else f"{fv:.2f}")
    ax.text(x["fdr"], y, ftxt, fontsize=FD, ha="right", va="center", fontweight="bold" if sigF else "normal",
            color=ctc if sigF else "#333")
    # I²
    i2=row["I2"]; itxt="NA" if pd.isna(i2) else f"{i2:.0f}%"
    icol="#4CAF50" if (not pd.isna(i2) and i2<25) else "#FFC107" if (not pd.isna(i2) and i2<50) else "#FF9800" if (not pd.isna(i2) and i2<75) else "#f44336"
    ax.text(x["i2"], y, itxt, fontsize=FD, ha="right", va="center", color=icol if not pd.isna(i2) else "#aaa", fontweight="bold")

# ── x-axis ────────────────────────────────────────────────────────────────────
y_axis=-0.55
ax.plot([x["fL"],x["fR"]],[y_axis,y_axis],"k-",lw=1)
ax.axvline(to_x(0),color="#888",ls="--",lw=0.8,ymin=0.06,ymax=0.92,alpha=0.6)
for orv in [0.5,0.75,1.0,1.5,2.0]:
    lv=np.log(orv)
    if lo_b<=lv<=hi_b:
        tx=to_x(lv); ax.plot([tx,tx],[y_axis,y_axis-0.09],"k-",lw=0.8)
        ax.text(tx,y_axis-0.15,f"{orv:g}",fontsize=FS,ha="center",va="top")
ax.text((x["fL"]+x["fR"])/2, y_axis-0.5, "Burden OR per decade (log scale)", fontsize=FD, fontweight="bold", ha="center")

# ── legend + title ────────────────────────────────────────────────────────────
leg=[Line2D([0],[0],marker=COHORT_MARK[COHORTS[c]["name"]],color="w",markerfacecolor=COHORT_COL[COHORTS[c]["name"]],
            markersize=5,markeredgecolor="black",markeredgewidth=0.3,label=COHORTS[c]["name"]) for c in cohort_list]
leg += [Line2D([0],[0],marker="D",color="w",markerfacecolor="#666",markersize=6,markeredgecolor="black",label="Pooled (RE)")]
ax.legend(handles=leg, loc="lower right", fontsize=FS, frameon=True, framealpha=0.95, edgecolor="#ccc",
          ncol=3, bbox_to_anchor=(0.99,0.0), handletextpad=0.3, columnspacing=0.6)
ax.text(0.5, y_clbl+0.45, f"Senescence-burden meta-analysis — {CONDITION_TAG} ({len(cohort_list)} cohorts)",
        fontsize=FH+1, fontweight="bold", ha="center")
ax.text(0.005, y_axis-0.5, "★ FDR<0.05   * p<0.05", fontsize=FS, ha="left", va="top", color="#666")

ax.set_xlim(-0.01,1.01); ax.set_ylim(-1.3, y_clbl+0.75); ax.axis("off")
plt.tight_layout()
fig.savefig(FIGDIR/f"meta_{EFFECT}_{CONDITION_TAG}_forest.pdf", dpi=300, bbox_inches="tight", facecolor="white")
fig.savefig(FIGDIR/f"meta_{EFFECT}_{CONDITION_TAG}_forest.svg", dpi=300, bbox_inches="tight", facecolor="white")
fig.savefig(FIGDIR/f"meta_{EFFECT}_{CONDITION_TAG}_forest.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(f"\n  ✓ saved → meta_{EFFECT}_{CONDITION_TAG}_forest.{{pdf,svg,png}}")

---
## Interpreting the output

Three questions, in this order.

**Did it replicate?** Look at the per-cohort markers before the diamond. Two
intervals overlapping on the same side of 1 is replication. A pooled estimate
away from 1 built from one strong cohort and one null cohort is not, however
small the pooled p.

**Is the pooling honest?** Check I² and the Q p-value. High heterogeneity means
the cohorts are estimating different things and the pooled number is an average
over that disagreement, not a better estimate of one quantity.

**Burden or susceptibility?** This module pools whichever file `SLUG` points at.
A cell type that moves on burden may be doing so because it is becoming more
abundant, not more senescent — module 06 sections 31 and 33 separate the two,
and the answer there governs how a pooled burden OR should be read.

**Not in this module:** any pooling of susceptibility, disease-mode contrasts,
or effects other than the per-decade burden GLMM. All are reachable by changing
`COHORTS`, `STUDY_TYPE`, `EFFECT` **and** `SLUG` together — see the note in
section 01.